In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq
from scipy.interpolate import interp1d
from scipy.signal import find_peaks, welch
import os
from scipy.optimize import curve_fit
import time

plt.rcParams['figure.dpi'] = 400


def load_files(integer):
    folder_path = "simulation_results_no_chirp"
    file_extension = f"_{integer}.npy"
    date_time_str = ""

    for filename in os.listdir(folder_path):
        if filename.endswith(file_extension):
            # Extract the date and time part from the filename
            parts = filename.split('_')
            date_time_str = f"{parts[1]}_{parts[2]}_{parts[3].split('.')[0]}"

            if filename.startswith("params"):
                theta = np.load(os.path.join(folder_path, filename))
            elif filename.startswith("time_data"):
                t = np.load(os.path.join(folder_path, filename))
            elif filename.startswith("voltage_data"):
                v = np.load(os.path.join(folder_path, filename))
            elif filename.startswith("current_data"):
                current = np.load(os.path.join(folder_path, filename))
    return theta, t, v, current, date_time_str




for integer in range(1000):
    theta, t, v, current, date_time_str = load_files(integer)
    p1, p2, p3, p4, p5, p6, p7, p8 = theta[9:17]

    start_time=time.time()

    # Interpolate the entire dataset
    #num_points = 5000  # Number of points for interpolation
    #min_time_step = np.min(np.diff(t))

    #num_points = len(t)
    num_points = 2505*35;
    t_interp = np.linspace(t[0], t[-1], num_points)
    #t_interp = np.arange(t[0], t[-1] + min_time_step, min_time_step)
    #num_points = int((t[-1] - t[0]) / min_time_step) + 1

    #t_interp = np.linspace(t[0], t[-1], num_points)
    #print(f"Number of interpolation points: {num_points}")
    #print(len(t_interp))
    v_interp_func = interp1d(t, v, kind='linear')
    i_interp_func = interp1d(t, current, kind='linear')

    v_interp = v_interp_func(t_interp)
    i_interp = i_interp_func(t_interp)

    # Break the interpolated data into blocks
    block_durations = [200,p2, p3, p5, p6, p5, 300]
    full_durations = block_durations * 5  # Adjust the repetition count as needed

    start_times = [0]
    for duration in full_durations[:-1]:  # Exclude the last duration to avoid going out of bounds
        next_start_time = start_times[-1] + duration
        start_times.append(next_start_time)
    end_times = [start + duration for start, duration in zip(start_times, full_durations)]

    def find_index(time_array, time):
        return np.searchsorted(time_array, time, side='right')

    start_indices = [find_index(t_interp, time) for time in start_times]
    end_indices = [find_index(t_interp, time) - 1 for time in end_times]  # -1 because end index is inclusive

    start_indices[0] = 0

    tBlocks = [t_interp[start:end+1] for start, end in zip(start_indices, end_indices)]  # +1 because end index is inclusive
    vBlocks = [v_interp[start:end+1] for start, end in zip(start_indices, end_indices)]
    iBlocks = [i_interp[start:end+1] for start, end in zip(start_indices, end_indices)]

    # Print the blocks
    # print(f"Total blocks created: {len(tBlocks)}")
    # for i, block in enumerate(tBlocks):
    #     if len(block) > 0:
    #         print(f"Block {i+1}: Start time = {block[0]}, End time = {block[-1]}, Duration = {block[-1] - block[0]} seconds, Length = {len(block)}")
    #     else:
    #         print(f"Block {i+1} is empty!")

    interp_tBlocks = tBlocks
    interp_vBlocks = vBlocks
    interp_iBlocks = iBlocks

    def exponential_decay(t, k):
        return np.exp(-k * t)
    def negative_exponential_decay(t, k):
        return -np.exp(-k * t)
    def process_special_block(v_block, t_block):
        # First voltage value (V0) is just the first value in v_block
        # Perform the curve fitting
        # def fit_exponential_decay(t, V):
        #     V0 = V[0]
        #     t_values = t - t[0]  # Time differences relative to the first time point
        #     v_values = V / V0  # Normalize by V0
        #     popt, _ = curve_fit(exponential_decay, t_values, v_values)
        #     k = popt[0]
        #     return V0, k
        # V0, k = fit_exponential_decay(t_block,v_block)
        V0 = v_block[0]

        # Fit the difference to an exponential decay V = V0 * exp(-k * t) to find k
        #t_values = t_block[:2] - t_block[0]  # Time differences relative to the first time point
        #v_values = v_block[:2] / V0  # Normalize by V0
        #popt, _ = curve_fit(exponential_decay, t_values, v_values)
        #k = popt[1]
        #print("popt length", len(popt))
        #print("popt:", popt)
        #print("vblock [1]", v_block[1])
        #print("vblock[0]", v_block[0])
        k = ( (v_block[1] - v_block[0]) / (t_block[1] - t_block[0]) )
        k = k / V0#because when you linearize the slope has a V0 term to it 
        #print("k", k)

        # Find the final voltage value in the block
        final_V = v_block[-1]

        return V0, k, final_V

    def process_exponential_decay_block(v_block, t_block):
        #num_points = int(np.ceil(len(v_block) / 4))
        #print("This is for the block 3 calculation")
        #def fit_exponential_decay(t, V, is_positive_decay):
            #V0 = V[0]
            #t_values = t - t[0]  # Time differences relative to the first time point
            #v_values = V / V0  # Normalize by V0
            #t_values = t - t[0]
            #v_values = V
            #popt, _ = curve_fit(exponential_decay, t_values, v_values)
            #initial_guess = [1e-5]
            #bounds = (0, 1)
            #if is_positive_decay:
            #    popt, pcov = curve_fit(exponential_decay, t_values, v_values, p0=initial_guess, bounds=bounds)
            #else:
            #    popt, pcov = curve_fit(negative_exponential_decay, t_values, v_values, p0=initial_guess, bounds=bounds)

            #popt, pcov = curve_fit(exponential_decay, t_values, v_values, p0=initial_guess, bounds=bounds)
            #k = popt[0]
            #return k, pcov

        #print("This is the t_values:", )
        V0 = v_block[0]
        #t_values = t_block[:num_points]  # Time differences relative to the first time point
        #print("This is the t_values:", t_values)
        #v_values = v_block[:num_points]
        final_V = v_block[-1]
        is_positive_decay = final_V < V0 
        #print("This is the v_values:", v_values)
        #k, pcov = fit_exponential_decay(t_values,v_values, is_positive_decay)
        #popt, _ = curve_fit(exponential_decay, t_values, v_values)
        #k = popt[0]

        return V0, k, final_V

    def calculate_summary_statistics(t_block, v_block, i_block):
        # Calculate resistance
        resistance = np.abs( (v_block[0] - v_block[-1]) / np.mean(i_block))

        # Calculate dV/dt
        dV_dt = np.gradient(v_block, t_block)
        #dV_dt = np.mean(dV_dt)
        # Calculate charge (Q) by integrating current over time
        Q = np.cumsum(-i_block) * (t_block[1] - t_block[0])

        # Calculate dV/dQ
        dV_dQ = np.gradient(v_block, Q)
        #dV_dQ = np.mean(dV_dQ)

        return resistance, np.mean(dV_dt), np.mean(dV_dQ), Q



    results_blocks = []
    results_array = []

    for blockNumber, (t_block, v_block, i_block) in enumerate(zip(interp_tBlocks, interp_vBlocks, interp_iBlocks)):

        if blockNumber % 7 == 0:
            V0, k, final_V = process_special_block(v_block, t_block)
            result = {"V0": V0, "k": k, "final_V": final_V}
            results_blocks.append((blockNumber, result))
            results_array.append(result)
        elif blockNumber % 7 == 1:
            resistance, dV_dt, dV_dQ, Q = calculate_summary_statistics(t_block, v_block, i_block)
            result = {
                "resistance": resistance,
                "dV_dt": dV_dt,
                "dV_dQ": dV_dQ
            }
            results_blocks.append((blockNumber, result))
            results_array.append(result)
        elif blockNumber % 7 == 2:
            V0, k, final_V = process_special_block(v_block, t_block)
            result = {"V0": V0, "k": k, "final_V": final_V}
            results_blocks.append((blockNumber, result))
            results_array.append(result)
        elif blockNumber % 7 == 3:
            resistance, dV_dt, dV_dQ, Q = calculate_summary_statistics(t_block, v_block, i_block)
            result = {
                "resistance": resistance,
                "dV_dt": dV_dt,
                "dV_dQ": dV_dQ
            }
            results_blocks.append((blockNumber, result))
            results_array.append(result)
        elif blockNumber % 7 == 4:
            V0, k, final_V = process_special_block(v_block, t_block)
            result = {"V0": V0, "k": k, "final_V": final_V}
            results_blocks.append((blockNumber, result))
            results_array.append(result)
        elif blockNumber % 7 == 5:
            resistance, dV_dt, dV_dQ, Q = calculate_summary_statistics(t_block, v_block, i_block)
            result = {
                "resistance": resistance,
                "dV_dt": dV_dt,
                "dV_dQ": dV_dQ
            }
            results_blocks.append((blockNumber, result))
            results_array.append(result)
        elif blockNumber % 7 == 6:
            V0, k, final_V = process_special_block(v_block, t_block)
            result = {"V0": V0, "k": k, "final_V": final_V}
            results_blocks.append((blockNumber, result))
            results_array.append(result)



    nu_array = []
    for blockNumber, results in results_blocks:

        if blockNumber % 7 == 0:
            #print(f"This is block: {blockNumber}")

            # print(f"Block Number: {blockNumber}")
            # print(f"  V0: {results['V0']:.3f}")
            nu_array.append(results['V0'])
            nu_array.append(results['k'])
            nu_array.append(results['final_V'])
            # print(f"  k: {results['k']:.9f}")
            # print(f"  Final V: {results['final_V']:.3f}")
            values = [results['V0'], results['k'], results['final_V']]
            #print(f"This is block: {blockNumber}")
        elif blockNumber % 7 == 1:
            #print(f"This is block: {blockNumber}")

            # print(f"Block Number: {blockNumber}")
            # print(f"  Resistance: {results['resistance']:.3f} Ohms")
            # print(f"  dV/dt: {results['dV_dt']}")
            # print(f"  dV/dQ: {results['dV_dQ']}")
            values = [results['resistance'], results['dV_dt'], results['dV_dQ']]
            nu_array.append(results['resistance'])
            nu_array.append(results['dV_dt'])
            nu_array.append(results['dV_dQ'])
            #print(f"This is block: {blockNumber}")
        elif blockNumber % 7 == 2:
            #print(f"This is block: {blockNumber}")

            #print(f"Block Number: {blockNumber}")
            #print(f"  V0: {results['V0']:.3f}")
            #print(f"  k: {results['k']:.9f}")
            #print(f"  Final V: {results['final_V']:.3f}")
            values = [results['V0'], results['k'], results['final_V']]
            nu_array.append(results['V0'])
            nu_array.append(results['k'])
            nu_array.append(results['final_V'])
            #print(f"This is block: {blockNumber}")
        elif blockNumber % 7 == 3:
            #print(f"This is block: {blockNumber}")

            #print(f"Block Number: {blockNumber}")
            #print(f"  Resistance: {results['resistance']:.3f} Ohms")
            #print(f"  dV/dt: {results['dV_dt']}")
            #print(f"  dV/dQ: {results['dV_dQ']}")
            values = [results['resistance'], results['dV_dt'], results['dV_dQ']]
            nu_array.append(results['resistance'])
            nu_array.append(results['dV_dt'])
            nu_array.append(results['dV_dQ'])
            #print(f"This is block: {blockNumber}")
        elif blockNumber % 7 == 4:
            #print(f"This is block: {blockNumber}")

            # print(f"Block Number: {blockNumber}")
            # print(f"  V0: {results['V0']:.3f}")
            # print(f"  k: {results['k']:.9f}")
            # print(f"  Final V: {results['final_V']:.3f}")
            values = [results['V0'], results['k'], results['final_V']]
            nu_array.append(results['V0'])
            nu_array.append(results['k'])
            nu_array.append(results['final_V'])
            #print(f"This is block: {blockNumber}")
        elif blockNumber % 7 == 5:
            #print(f"This is block: {blockNumber}")

            # print(f"Block Number: {blockNumber}")
            # print(f"  Resistance: {results['resistance']:.3f} Ohms")
            # print(f"  dV/dt: {results['dV_dt']}")
            # print(f"  dV/dQ: {results['dV_dQ']}")
            values = [results['resistance'], results['dV_dt'], results['dV_dQ']]
            nu_array.append(results['resistance'])
            nu_array.append(results['dV_dt'])
            nu_array.append(results['dV_dQ'])
            #print(f"This is block: {blockNumber}")
        elif blockNumber % 7 == 6:
            #print(f"This is block: {blockNumber}")

            # print(f"Block Number: {blockNumber}")
            # print(f"  V0: {results['V0']:.3f}")
            # print(f"  k: {results['k']:.15f}")
            # print(f"  Final V: {results['final_V']:.3f}")
            nu_array.append(results['V0'])
            nu_array.append(results['k'])
            nu_array.append(results['final_V'])
            values = [results['V0'], results['k'], results['final_V']]
            #print(f"This is block: {blockNumber}")
    nu_array_np = np.array(nu_array)
    folder_path = "summary_statistics_fixed_design_no_chirp"
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

    # Define the new file name based on the extracted date, time, and integer value
    file_name = f"summary_stats_{date_time_str}_{integer}.npy"
    file_path = os.path.join(folder_path, file_name)

    # Save the numpy array to the file
    np.save(file_path, nu_array_np)

    print(f"Array saved to {file_path}")

    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"Elapsed time for integer {integer}: {elapsed_time} seconds")

Array saved to summary_statistics_fixed_design_no_chirp/summary_stats_20240518_165456_0_0.npy
Elapsed time for integer 0: 0.009475946426391602 seconds
Array saved to summary_statistics_fixed_design_no_chirp/summary_stats_data_20240518_165458_1.npy
Elapsed time for integer 1: 0.007330894470214844 seconds
Array saved to summary_statistics_fixed_design_no_chirp/summary_stats_data_20240518_165458_2.npy
Elapsed time for integer 2: 0.006842136383056641 seconds
Array saved to summary_statistics_fixed_design_no_chirp/summary_stats_data_20240518_165458_3.npy
Elapsed time for integer 3: 0.008488655090332031 seconds
Array saved to summary_statistics_fixed_design_no_chirp/summary_stats_20240518_165458_4_4.npy
Elapsed time for integer 4: 0.007487297058105469 seconds
Array saved to summary_statistics_fixed_design_no_chirp/summary_stats_data_20240518_165504_5.npy
Elapsed time for integer 5: 0.006765127182006836 seconds
Array saved to summary_statistics_fixed_design_no_chirp/summary_stats_20240518_165